# Simulation Reproducibility

This notebook summarizes the simulation outputs in `results/simulation/`. The expensive simulation runs are handled by scripts in `reproducibility/simulation/`; this notebook loads existing result files, displays compact data frames, and exports LaTeX tables.

In [1]:
from pathlib import Path
import os

if Path.cwd().name == "reproducibility":
    os.chdir("..")

import pandas as pd

from reproducibility.simulation.simulation_tables import (
    load_subspace_results,
    load_prediction_results,
    export_subspace_table,
    export_prediction_table,
)

pd.set_option("display.max_rows", 100)
RESULT_ROOT = Path("results/simulation")
TABLE_DIR = Path("results/tables")


def format_mean_se(mean, se, digits=3, omit_leading_zero=False):
    if pd.isna(mean):
        return "--"
    text = f"{mean:.{digits}f} ({se:.{digits}f})"
    return text.replace("0.", ".") if omit_leading_zero else text


## Run Scripts

Run these from the repository root when regenerating results.

CKDR/SCA simulations:

```powershell
foreach ($n in 200,500,1000) {
  foreach ($y in "Y1","Y2","Y3","Y4") {
    foreach ($m in "oracle","cv") {
      python reproducibility/simulation/CKDR_simulation.py --n $n --y_func $y --m $m --n_jobs -2
    }
  }
}
```

Python competitor prediction results:

```powershell
foreach ($method in "lc_lasso","clr_kernel","clr_rf") {
  foreach ($n in 200,500,1000) {
    foreach ($y in "Y1","Y2","Y3","Y4") {
      python reproducibility/simulation/python_competitor_predictions.py --method $method --n $n --y_func $y --n_jobs -2
    }
  }
}
```

Amalgam subspace results for classification settings:

```powershell
foreach ($n in 200,500,1000) {
  foreach ($y in "Y3","Y4") {
    python reproducibility/simulation/amalgam_subspace.py --n $n --y_func $y --n_jobs -2
  }
}
```

RS-ES results are generated by the MATLAB script `reproducibility/other_methods/Relative-shift/JP_simulation_RSES.m`.

## Subspace Recovery

In [2]:
subspace_df = load_subspace_results()

subspace_display = subspace_df.copy()
subspace_display["rho"] = subspace_display.apply(
    lambda row: format_mean_se(row["rho_mean"], row["rho_se"], digits=1), axis=1
)
subspace_display["ARI"] = subspace_display.apply(
    lambda row: format_mean_se(row["ari_mean"], row["ari_se"], digits=1), axis=1
)

subspace_table = pd.concat(
    [
        subspace_display.pivot_table(
            index=["setting", "method"], columns="n", values="rho", aggfunc="first"
        ),
        subspace_display.pivot_table(
            index=["setting", "method"], columns="n", values="ARI", aggfunc="first"
        ),
    ],
    axis=1,
    keys=[r"rho x 100", "ARI x 100"],
)

subspace_rows = []
for setting, methods in {
    "(i)": [r"CKDR-$m^\star$", r"CKDR$^*$", "RS-ES"],
    "(ii)": [r"CKDR-$m^\star$", r"CKDR$^*$", "RS-ES"],
    "(iii)": [r"CKDR-$m^\star$", r"CKDR$^*$", "Amalgam"],
    "(iv)": [r"CKDR-$m^\star$", r"CKDR$^*$", "Amalgam"],
}.items():
    subspace_rows.extend((setting, method) for method in methods)
subspace_table = subspace_table.reindex(pd.MultiIndex.from_tuples(subspace_rows, names=["setting", "method"]))
subspace_table


rho x 100                           ARI x 100  \
n                             200         500         1000        200    
setting method                                                           
(i)     CKDR-$m^\star$  13.1 (0.4)   7.5 (0.6)   4.2 (0.0)  98.0 (0.7)   
        CKDR$^*$        11.9 (0.4)   6.4 (0.1)   4.4 (0.0)  53.6 (2.3)   
        RS-ES           12.8 (0.1)   6.5 (0.1)   4.3 (0.0)  99.5 (0.1)   
(ii)    CKDR-$m^\star$  56.3 (0.4)  44.1 (0.7)  31.4 (0.8)  62.0 (1.7)   
        CKDR$^*$        55.9 (0.4)  45.0 (0.7)  32.7 (0.7)  62.7 (1.5)   
        RS-ES                   --          --          --  55.9 (1.4)   
(iii)   CKDR-$m^\star$  35.7 (0.3)  18.7 (0.2)  12.2 (0.1)  45.5 (0.7)   
        CKDR$^*$        56.4 (1.0)  29.6 (1.7)  12.3 (0.7)  41.1 (1.0)   
        Amalgam         56.2 (0.4)  42.5 (0.5)  35.8 (0.4)  20.2 (0.5)   
(iv)    CKDR-$m^\star$  64.4 (0.2)  56.8 (0.3)  49.9 (0.6)  42.6 (0.8)   
        CKDR$^*$        74.8 (0.3)  68.0 (0.5)  53.1 (0.5)  42.2 (0.9)   
        Amalgam         75.1 (0.2)  70.4 (0.2)  66.6 (0.2)  17.4 (0.5)   

                                                
n                             500         1000  
setting method                                  
(i)     CKDR-$m^\star$  96.3 (1.4)  98.8 (0.8)  
        CKDR$^*$        37.7 (1.5)  49.9 (2.5)  
        RS-ES           97.6 (1.2)  98.8 (0.8)  
(ii)    CKDR-$m^\star$  82.0 (2.4)  88.2 (2.2)  
        CKDR$^*$        84.1 (2.2)  91.7 (1.8)  
        RS-ES           68.3 (2.1)  74.6 (2.3)  
(iii)   CKDR-$m^\star$  73.3 (0.9)  93.6 (0.5)  
        CKDR$^*$        69.6 (1.3)  79.1 (1.4)  
        Amalgam         34.2 (0.6)  40.6 (0.4)  
(iv)    CKDR-$m^\star$  66.9 (1.1)  76.9 (1.4)  
        CKDR$^*$        63.1 (1.2)  76.3 (1.4)  
        Amalgam         25.3 (0.6)  34.5 (0.9)

The table above has the same row and column layout as `results/tables/simulation_subspace.tex`.


In [3]:
subspace_df, subspace_tex = export_subspace_table()
print(f"Wrote {subspace_tex}")

Wrote results\tables\simulation_subspace.tex


## Prediction Performance

In [2]:
prediction_df = load_prediction_results()

prediction_display = prediction_df.copy()
prediction_display["value"] = prediction_display.apply(
    lambda row: format_mean_se(row["mean"], row["se"], digits=3, omit_leading_zero=True),
    axis=1,
)

prediction_methods = [
    r"CKDR-$m^\star$", r"CKDR$^*$", "LC-Lasso", "clr-Kernel", "clr-RF", "RS-ES"
]
prediction_table = prediction_display.pivot_table(
    index=["metric", "setting", "n"], columns="method", values="value", aggfunc="first"
).reindex(columns=prediction_methods)

prediction_rows = []
for metric, settings in [("MSE", ["(i)", "(ii)"]), ("MCR", ["(iii)", "(iv)"])]:
    for setting in settings:
        prediction_rows.extend((metric, setting, n) for n in [200, 500, 1000])
prediction_table = prediction_table.reindex(
    pd.MultiIndex.from_tuples(prediction_rows, names=["metric", "setting", "n"])
)
prediction_table


method              CKDR-$m^\star$     CKDR$^*$     LC-Lasso   clr-Kernel  \
metric setting n                                                            
MSE    (i)     200     .026 (.001)  .030 (.001)  .032 (.000)  .120 (.002)   
               500     .014 (.001)  .020 (.000)  .019 (.000)  .090 (.001)   
               1000    .012 (.000)  .014 (.000)  .017 (.000)  .078 (.000)   
       (ii)    200     .077 (.004)  .081 (.006)  .164 (.005)  .210 (.005)   
               500     .031 (.001)  .032 (.002)  .107 (.002)  .165 (.002)   
               1000    .022 (.001)  .023 (.001)  .099 (.002)  .144 (.002)   
MCR    (iii)   200     .156 (.003)  .161 (.003)  .229 (.004)  .223 (.004)   
               500     .088 (.001)  .090 (.002)  .191 (.002)  .179 (.002)   
               1000    .067 (.001)  .069 (.001)  .154 (.001)  .156 (.001)   
       (iv)    200     .180 (.003)  .179 (.003)  .286 (.004)  .241 (.004)   
               500     .114 (.002)  .116 (.002)  .212 (.002)  .203 (.002)   
               1000    .100 (.001)  .098 (.001)  .183 (.001)  .179 (.001)   

method                    clr-RF        RS-ES  
metric setting n                               
MSE    (i)     200   .316 (.004)  .020 (.000)  
               500   .281 (.002)  .012 (.000)  
               1000  .262 (.001)  .011 (.000)  
       (ii)    200   .345 (.008)  .130 (.004)  
               500   .315 (.004)  .096 (.002)  
               1000  .306 (.004)  .091 (.002)  
MCR    (iii)   200   .338 (.003)          NaN  
               500   .290 (.002)          NaN  
               1000  .258 (.002)          NaN  
       (iv)    200   .361 (.004)          NaN  
               500   .315 (.003)          NaN  
               1000  .284 (.001)          NaN

The table above has the same row and column layout as `results/tables/simulation_predictions.tex`.


In [3]:
prediction_df, prediction_tex = export_prediction_table()
print(f"Wrote {prediction_tex}")

Wrote results\tables\simulation_predictions.tex
